# 1. IMPORTACIÓN DE DATOS

In [ ]:
# Donde estoy

import os

os.getcwd()

In [ ]:
import time

In [ ]:
# ==================================
# 1. IMPORTACIÓN DE DATOS
# ==================================

# Importar los datos del fichero csv en un dataframe

import csv
import pandas as pd
pd.options.display.max_rows  # Para mostrar todas las columnas
pd.set_option('display.max_colwidth', -1)  # Para incluir todo el contenido de una celda, sin truncar contenido.
pd.set_option('display.max_columns', 500)  # Para incluir todas las columnas (no serán más de 500)
pd.set_option('display.max_rows', 6000)  # Para incluir todas las columnas (no serán más de 6000)

import csv
import pandas as pd
file = 'stsbenchmark\sts-train.csv'
corpus = pd.read_csv(file, sep='\t', error_bad_lines=False, quoting=csv.QUOTE_NONE, usecols=range(7), header=-1)
corpus.head()
corpus.columns = ['genre', 'file', 'year', 'id', 'gold_score', 'sent1', 'sent2']
corpus.sample(5)

# 2. Análisis 1: Revisión general de los datos
## Descripción básica
### Columnas
* 7 columnas: 
    * **genre** = tipo de contenido del que proceden los textos
        * Tipos: 3: ['main-captions' 'main-forum' 'main-news']
    * **file** = fichero de procedencia de los textos
        * Ficheros: 6: ['MSRvid' 'images' 'deft-forum' 'MSRpar' 'deft-news' 'headlines']
        * Cada fichero incluye un solo tipo de contenido:
            * 'main-captions': 'MSRvid', 'images'
            * 'main-forum': 'deft-forum'
            * 'main-news': MSRpar, 'deft-news', 'headlines']
    * **year** = año de los datos
        * Años: 5: 2012-2016 (para 2012 hay una subdivisión 2012test+2012train
    * **id** = identificador de un par de frases para un fichero y año.
        * No es único
            * La combinación para identificar parejas únicas es 'file'+'year'+'id'
        * No es consecutivo, faltan valores
    * **gold_score** = puntuación de similitud dada por humanos
        * Rango: [0-5]
    * **sent1** = texto (frase)
    * **sent2** = texto (frase)
    
### Filas
    
* 5749 filas (**5706** tras eliminar duplicados)
    * Clasificación por tipos de contenido:
        * 2000 "captions" (**1991**)
        * 450 "forum" (**423**)
        * 3299 "news" (**3292**)

### Valores nulos y duplicados
* No hay valores nulos
* Valores únicos:
    * Los id no son únicos
    * <span style="color:red">Combinación única: 'file'+'year'+'id'</span>
* Duplicados en las parejas de frases:
    * 227 duplicados de sent1
    * 243 duplicados de sent2
    * 39 duplicados de ambos sent1 y sent2 --> <span style='color:red'>SE ELIMINAN, dejando solo la primera instancia</span>



In [ ]:
# ==================================
# 2. ANÁLISIS PREVIO DE LOS DATOS
# ==================================

# 2.1 Descripción estadística básica

# Pandas describe()

include =['object', 'float', 'int'] 
corpus.describe(include = include).transpose()

In [ ]:
# Tipos y procedencia de los textos

print('Tipos de contenido (', len(corpus['genre'].unique()), ') :', corpus['genre'].unique())
print()
print('Ficheros de procedencia (', len(corpus['file'].unique()), ') :', corpus['file'].unique())
print()
print('Tipos de contenido / Ficheros de procedencia (', len((corpus['genre'] + ' - ' + corpus['file']).unique()), ') :', (corpus['genre'] + ' - ' + corpus['file']).unique())
print()
print('Años (', len(corpus['year'].unique()), ') :', sorted(corpus['year'].unique()))



In [ ]:
# Identificador único

duplicados_id_bool = corpus['id'].duplicated(keep = False)
duplicados_id = corpus[duplicados_id_bool].groupby('id').count()
num_duplicados_id = len(duplicados_id)
print('Número de ids repetidos dos o más veces:', num_duplicados_id)

duplicados_file_id_bool = corpus[['file', 'id']].duplicated(keep = False)
duplicados_file_id = corpus[duplicados_file_id_bool].groupby(['file', 'id']).count()
num_duplicados_file_id = len(duplicados_file_id )
print('Número de combinaciones file + ids repetidas dos o más veces:', num_duplicados_file_id)

duplicados_file_year_id_bool = corpus[['file', 'year', 'id']].duplicated(keep = False)
duplicados_file_year_id = corpus[duplicados_file_year_id_bool].groupby(['file', 'year', 'id']).count()
num_duplicados_file_year_id = len(duplicados_file_year_id )
print('Número de combinaciones file + ids repetidas dos o más veces:', num_duplicados_file_year_id)


In [ ]:
# Agregación de datos
'''
PENDIENTE: Fila SUBTOTALES en el sitio correcto
'''

resumen = corpus[['genre', 'file', 'id']].copy()
resumen = resumen.groupby(['genre', 'file']).count()

totales1 = resumen.groupby(['genre']).sum()
totales1.index = [totales1.index.get_level_values(0),
                ['SUBTOTAL'] * len(totales1)]

resumen = pd.concat([resumen, totales1]).sort_index(level=[0])

resumen = pd.DataFrame(resumen)
resumen


In [ ]:
# Valores nulos y duplicados de frases

print('VALORES NULOS EN CORPUS:')
print()
print(corpus.isna().any())  # En Pandas isna() y isnull() son exactamente iguales

duplicados1_bool = corpus['sent1'].duplicated(keep = False)
duplicados1 = corpus[duplicados1_bool].sort_values('sent1')
duplicados1_unicos = duplicados1[['sent1', 'id']].groupby('sent1').count().sort_values('id', ascending=False)

duplicados2_bool = corpus['sent2'].duplicated(keep = False)
duplicados2 = corpus[duplicados2_bool].sort_values('sent2')
duplicados2_unicos = duplicados2[['sent2', 'id']].groupby('sent2').count().sort_values('id', ascending=False)

duplicados12_bool = corpus[['sent1', 'sent2']].duplicated(keep = False)
duplicados12 = corpus[duplicados12_bool].sort_values('sent1')
duplicados12_unicos = duplicados12[['sent1','sent2', 'id']].groupby(['sent1', 'sent2']).count().sort_values('id', ascending=False)


print()
print('NÚMERO DE VALORES DUPLICADOS EN SENT1:', duplicados1_unicos.count())
print()
print('NÚMERO DE VALORES DUPLICADOS EN SENT2:', duplicados2_unicos.count())
print()
print('NÚMERO DE VALORES DUPLICADOS EN SENT1 Y SENT2:', duplicados12_unicos.count())
print()
print('VALORES DUPLICADOS EN SENT1:')
print(duplicados1_unicos)
print()
print('VALORES DUPLICADOS EN SENT2:')
print(duplicados2_unicos)
print()
print('VALORES DUPLICADOS EN SENT1 Y SENT2:')
print(duplicados12_unicos)



In [ ]:
# Eliminación de duplicados

corpus.drop_duplicates(['sent1', 'sent2'], keep='first', inplace=True)

In [ ]:
# Ejemplos de pares según su similitud

# Muy similares

corpus.sort_values('gold_score', ascending=False).head()


In [ ]:
# Muy distintos
corpus.sort_values('gold_score').head()


In [ ]:
# Creamos un corpus reducido con filas representativas del corpus original

 # Para realizar pre-procesamientos que nos permitan verificar la adecuación de los algoritmos
 # en un tiempo reducido.

c = corpus[['id', 'gold_score', 'sent1', 'sent2']].iloc[::100, :].copy()
comentarios = 'Número de filas incluídas en el dataframe: ' + str(c.shape[0])

print(comentarios)

c.head()

In [ ]:
# ==================================
# 2. ANÁLISIS PREVIO DE LOS DATOS
# ==================================

'''
Longitud máxima en términos de
- Caracteres
- Palabras
Histograma con frequencias de longitudes en terminos de
- Palabras
Wordnet (una vez tokenizado)
- Palabras que no están en Wordnet
    * Errores tipograficos: ej someoen
'''

# Stopwords que sacaremos:
'''
# nltk.corpus.stopwords.words('english')
- otras?
'''


# 3. TOKENIZACION, LIMPIEZA Y ANOTACIÓN


In [ ]:
# Pre procesamiento
# https://www.linkedin.com/pulse/processing-normalizing-text-data-saurav-ghosh/
# http://xperimentallearning.blogspot.com/2018/05/nlppython.html

# ========================================
# 3. TOKENIZACION, LIMPIEZA Y ANOTACIÓN
# ========================================

import nltk

def tokenizar(sent, info=False):
    
    # tokens que aceptaremos (de nltk.org/book/ch03). Output: cadena de palabras.
   
    if info == True: print('Frase a tokenizar:', sent)
        
    pattern = r'''(?x)       # set flag to allow verbose regexps
        (?:[A-Z]\.)+         # abbreviations, e.g. U.S.A. ?: needs to be added to specify that the parenthesis specify the scope of the disjunction, not the selection o strings to be extracted
       | \w+(?:-\w+)*        # words with optional internal hyphens
       | [\$,\€]?\d+(?:\.\d+)?%?  # currency and percentages, e.g. $12.40, 82%
    '''
    if info == True: print('pattern:', pattern)
    '''
    AMPLIAR.
    - Analizar errores
    - Alguna solución para detectar phrasal verbs?
    '''

    # tokenización. Output: lista
   
    tokens = nltk.regexp_tokenize(sent, pattern, gaps=False)
    
    # limpieza 1: mayúsculas
    
    tokens = [token.lower() for token in tokens]   
    if info == True: print('Tokens:', tokens)
 
    # anotación POS. Output: lista de tuplas
    
    tagged_tokens = nltk.pos_tag(tokens, tagset='universal')
    
    # Convertimos las etiquetas gramaticales en notación Lesk;
        
    for i, tagged_token in enumerate(tagged_tokens):
        if tagged_tokens[i][1] == 'NOUN': tagged_tokens[i] = (tagged_tokens[i][0], 'n')
        if tagged_tokens[i][1] == 'VERB': tagged_tokens[i] = (tagged_tokens[i][0], 'v')
        if tagged_tokens[i][1] == 'ADJ': tagged_tokens[i] = (tagged_tokens[i][0], 'a')
        if tagged_tokens[i][1] == 'ADV': tagged_tokens[i] = (tagged_tokens[i][0], 'r')
    if info == True: print('Tagged_tokens:', tagged_tokens)
    
    # limipieza 2: stopwords - REVISAR si queremos/podemos incluir phrasal verbs, determinantes, pronombres...
            
                # Wordnet no incluye determinantes o preposiciones, de forma que "a" es asociado por Lesk 
                # al Synset('deoxyadenosine_monophosphate.n.01'), lo cual no nos conviene.
                # Por otro lado, Lesk utiliza para desambiguar solo las palabras que aparecen en Wordnet,
                # de modo que limpiar antes o después de Lesk las stopwords no debería afectar al resultado.
   
    if info == True: print('Limpieza de stopwords')
    stopwords = nltk.corpus.stopwords.words('english')
    tagged_stops = [tagged_stop for tagged_stop in tagged_tokens if tagged_stop[0] in stopwords]
    if info == True: print('Stopwords eliminadas:', tagged_stops)
    tokens = [token for token in tokens if token not in stopwords]
    tagged_tokens = [tagged_token for tagged_token in tagged_tokens if tagged_token[0] not in stopwords]
    if info == True: print('Tagged tokens sin stopwords:', tagged_tokens)

    return(tokens, tagged_tokens, tagged_stops)

# Ejemplo para comprobar funcionamiento tokenizar()
   
sent = "I'll always love my sweet baby"
print('****** EJEMPLO tokenizar() ********')
tokens = tokenizar(sent, True)
print('****')


In [ ]:
# Tokenización de c

c['tokens1'], c['tokens2'] = '', ''
c['tagged_tokens1'], c['tagged_tokens2'] = '', ''
c['stops1'], c['stops2'] = '', ''

for i in c.index:
    c.at[i, 'tokens1'] = tokenizar(c.at[i, 'sent1'])[0]
    c.at[i, 'tokens2'] = tokenizar(c.at[i, 'sent2'])[0]
    c.at[i, 'tagged_tokens1'] = tokenizar(c.at[i, 'sent1'])[1]
    c.at[i, 'tagged_tokens2'] = tokenizar(c.at[i, 'sent2'])[1]
    c.at[i, 'stops1'] = tokenizar(c.at[i, 'sent1'])[2]
    c.at[i, 'stops2'] = tokenizar(c.at[i, 'sent2'])[2]
    
c.head()
c['tagged_tokens1'][0]


In [ ]:
# Tokenización de corpus
'''
corpus['tokens1'], corpus['tokens2'] = '', ''
corpus['tagged_tokens1'], corpus['tagged_tokens2'] = '', ''
corpus['stops1'], corpus['stops2'] = '', ''

for i in corpus.index:
    corpus.at[i, 'tokens1'] = tokenizar(corpus.at[i, 'sent1'])[0]
    corpus.at[i, 'tokens2'] = tokenizar(corpus.at[i, 'sent2'])[0]
    corpus.at[i, 'tagged_tokens1'] = tokenizar(corpus.at[i, 'sent1'])[1]
    corpus.at[i, 'tagged_tokens2'] = tokenizar(corpus.at[i, 'sent2'])[1]
    corpus.at[i, 'stops1'] = tokenizar(corpus.at[i, 'sent1'])[2]
    corpus.at[i, 'stops2'] = tokenizar(corpus.at[i, 'sent2'])[2]
'''

In [ ]:
corpus.sample(n=5)

In [ ]:
# ========================================
# 4. ASIGNACIÓN DE SYNSETS DE WORDNET
# ========================================

# Asignación de synsets - Se asigna el primer synset (el más común)
'''
HACER, ya que parece que Lesk no está funcionando muy bien.
'''

# Asignación de synsets 
    # Entrada: lista de tuplas con tokens y sus correspondientes etiquetas gramaticales
    # Argumentos: 
         # lesk: Opción de aplicar el algoritmo de Lesk  [Lesk 1986] de la librería wsd para desambiguar
                # Opción por defecto: n (no) --> Se asignará el primer synset del grupo de synsets (el más común)
                # Opción: s (sí) --> Asignar el synset determinado por Lesk
                
from nltk.corpus import wordnet as wn
                
lesk = input('¿Utilizar el desambiguador de Lesk para asignar synsets? (s/n)')

def syns(tagged_tokens, lesk='s'):
    from nltk.wsd import lesk
    for i in c.index:
        syns = list()
        errors = list()
        if lesk == 's':
            for j in tagged_tokens:
                if lesk([token for token, tag in tagged_tokens], j[0], j[1]) is not None:
                    syns.append(lesk([token for token, tag in tagged_tokens], j[0], j[1]))
                else:
                    errors.append(j)
        else: 
            for j in tagged_tokens:
                try: syns.append(wn.synsets(j[0],j[1])[0])
                except: errors.append(j)
        return(syns, errors)
    
# Ejemplo para comprobar funcionamiento syns()

tagged_tokens = c['tagged_tokens1'][0]
syns1 = syns(tagged_tokens)
%timeit -n 1 -r 1 syns1 = syns(tagged_tokens)

print('****** EJEMPLO syns() ********')   
print('tagged_tokens:', tagged_tokens)
print('syns(tagged_tokens):', syns1[0])
print('syns(tagged_tokens) - Errors:', syns1[1])
print('****')


In [ ]:
# Asignacion de synsets a los tokens de c

lesk = input('¿Utilizar el desambiguador de Lesk para asignar synsets? (s/n)')

c['syns1'] = ''
c['syns2'] = ''
c['errors1'] = ''
c['errors2'] = ''

for i in c.index:
    c.at[i,'syns1'] = syns(c.at[i, 'tagged_tokens1'])[0]
    c.at[i,'syns2'] = syns(c.at[i, 'tagged_tokens2'])[0]
    c.at[i,'errors1'] = syns(c.at[i, 'tagged_tokens1'])[1]
    c.at[i,'errors2'] = syns(c.at[i, 'tagged_tokens2'])[1]
    
c

# 4.Analisis 2: Tipo y distribución de tokens y synsets


In [ ]:
# Distribución del número de tokens en las frases a estudiar
    # Entrada: listas de tokens a analizar
    # Argumentos: 
        # leyenda = []   (lista de textos que queremos como leyenda de cada entrada)
        
from numpy import *
import math
import matplotlib.pyplot as plt

def distribucion3(*conjunto, leyendas=''):
    
    fig, axis = plt.subplots()

    axis.yaxis.grid(True)
    axis.set_title('Número de tokens')
    axis.set_xlabel('Número de tokens por frase')
    axis.set_ylabel('Número de frases')
    
    leyendas_plot = list()
            
    i = 0
    
    for tokens in conjunto:
                
        try:
            label = leyendas[i]
        except:
            label = 'Conjunto ' + str(i + 1)
        print('******',label.upper(), '******')
        
        distribucion = nltk.FreqDist(tokens)
        print('Longitudes:', tokens)
        print('Distribución:', distribucion)
        print(distribucion.tabulate())

        dist = nltk.FreqDist(tokens).items()
        dist = [item for item in dist]
        dist.sort()

        x = [x for x, y in dist ]
        y = [y for x, y in dist ]
        media = sum([x * y for x, y in dist ]) / sum(y)

        print('Número medio de tokens por frase:', media)
        print('y (número de frases):', y)
        print('x (número de tokens:', x)
        print('sum y:', sum(y))
        print('sum x*y:', sum([x * y for x, y in dist ]))
        
        axis.plot(x, y, label=label)
                
        i += 1

        handles, labels = axis.get_legend_handles_labels()
    axis.legend(handles, labels)


In [ ]:
# Distribución en c: TODOS

longitudes_con_stopwords = [len(tokens) for tokens in c['tokens1'] + c['stops1']] + [len(tokens) for tokens in c['tokens2'] + c['stops2']] 
longitudes_sin_stopwords = [len(tokens) for tokens in c['tokens1']] + [len(tokens) for tokens in c['tokens2']]

print('DISTRIBUCIÓN DE TOKENS: TODOS')
print()
distribucion3(longitudes_con_stopwords, longitudes_sin_stopwords, leyendas=['longitudes_con_stopwords', 'longitudes_sin_stopwords'] )


In [ ]:
# Distribución en c: por CATEGORÍAS

c['nombres1'] = ''
c['nombres2'] = ''
c['verbos1'] = ''
c['verbos2'] = ''  
c['adjetivos1'] = ''
c['adjetivos2'] = ''  
c['adverbios1'] = ''
c['adverbios2'] = ''  

for i in c.index:
    c.at[i, 'nombres1'] = [tagged_token for tagged_token in c.at[i, 'tagged_tokens1'] if tagged_token[1] == 'n']
    c.at[i, 'nombres2'] = [tagged_token for tagged_token in c.at[i, 'tagged_tokens2'] if tagged_token[1] == 'n']
    c.at[i, 'verbos1'] = [tagged_token for tagged_token in c.at[i, 'tagged_tokens1'] if tagged_token[1] == 'v']
    c.at[i, 'verbos2'] = [tagged_token for tagged_token in c.at[i, 'tagged_tokens2'] if tagged_token[1] == 'v']
    c.at[i, 'adjetivos1'] = [tagged_token for tagged_token in c.at[i, 'tagged_tokens1'] if tagged_token[1] == 'a']
    c.at[i, 'adjetivos2'] = [tagged_token for tagged_token in c.at[i, 'tagged_tokens2'] if tagged_token[1] == 'a']
    c.at[i, 'adverbios1'] = [tagged_token for tagged_token in c.at[i, 'tagged_tokens1'] if tagged_token[1] == 'r']
    c.at[i, 'adverbios2'] = [tagged_token for tagged_token in c.at[i, 'tagged_tokens2'] if tagged_token[1] == 'r']
    
longitudes_n = [len(tokens) for tokens in c['nombres1']] + [len(tokens) for tokens in c['nombres2']] 
longitudes_v = [len(tokens) for tokens in c['verbos1']] + [len(tokens) for tokens in c['verbos2']]
longitudes_a = [len(tokens) for tokens in c['adjetivos1']] + [len(tokens) for tokens in c['adjetivos2']]
longitudes_r = [len(tokens) for tokens in c['adverbios1']] + [len(tokens) for tokens in c['adverbios2']]

print('DISTRIBUCIÓN DE TOKENS: NOMBRES, VERBOS, ADJETIVOS Y ADVERBIOS')
print()
distribucion3(longitudes_n, longitudes_v, longitudes_a, longitudes_r, leyendas=['longitudes_n', 'longitudes_v', 'longitudes_a', 'longitudes_r'] )


In [ ]:
# ========================================
# 5. BAG OF WORDS
# ========================================

# Creación de la Bag of Words (BoW) (de hecho, será una "Bag of synsets")
    # La necesitaremos luego como espacio vectorial para aplicar el método [Liu 2013] de similitud a nivel de frase.

c['bow'] = c['syns1'] + c['syns2']

for i in c.index:
    c.at[i, 'bow'] = list(set(c.at[i, 'bow']))

# BoW de nombres y verbos

c['bow_nv'] = ''

for i in c.index:
    c.at[i, 'bow_nv'] = [synset for synset in c.at[i, 'bow'] if synset.pos() in ['n', 'v']] 


c.head()

# Similitud a nivel léxico: Métodos incluídos en Wordnet
    * Métodos:
        * ps --> path_similarity (POR DEFECTO). Rango: [0:1]
        * lch --> Leacock Chodorow. Rango: [0:3.64?] 
        * wup --> Wu_Palmer
        * res --> Resknik (requiere corpus ic)
        * jcn --> Jiang-Conrath (requiere corpus ic)
        * lin --> Lin (requiere corpus ic)
    * Consideraciones:
        * Todos estos métodos solo comparan NOMBRES y VERBOS, respectivamente.
            * Para adjetivos y adverbios, o para mezclas de nombres y verbos, dan error.

In [ ]:
# ========================================
# 6. SIMILITUD A NIVEL LÉXICO - MÉTODOS INCLUÍDOS EN WORDNET
# ========================================

# Ejemplos de funciones de similitud integradas en Wordnet

    # ps --> path_similarity (POR DEFECTO). Rango: [0:1]
    # lch --> Leacock Chodorow. Rango: [0:3.64?] 
    # wup --> Wu_Palmer
    # res --> Resknik (requiere corpus ic)
    # jcn --> Jiang-Conrath (requiere corpus ic)
    # lin --> Lin (requiere corpus ic)

from nltk.corpus import wordnet as wn

syn1 = wn.synset('dog.n.01')
syn2 = wn.synset('cat.n.01')
syn3 = wn.synset('black.a.01')
syn4 = wn.synset('white.a.01')

try: print('wn.path_similarity(', syn1, syn1, '):',  wn.path_similarity(syn1, syn1))
except Exception as e: print(e)
try: print('wn.path_similarity(', syn1, syn2, '):',  wn.path_similarity(syn1, syn2))
except Exception as e: print(e)
try: print('wn.path_similarity(', syn1, syn3, '):',  wn.path_similarity(syn1, syn3))
except Exception as e: print(e)
try: print('wn.path_similarity(', syn3, syn4, '):',  wn.path_similarity(syn3, syn4))
except Exception as e: print(e)
    
print()

try: print('wn.lch_similarity(', syn1, syn1, '):',  wn.lch_similarity(syn1, syn1))
except Exception as e: print(e)
try: print('wn.lch_similarity(', syn1, syn2, '):',  wn.lch_similarity(syn1, syn2))
except Exception as e: print(e)
try: rint('wn.lch_similarity(', syn1, syn3, '):',  wn.lch_similarity(syn1, syn3))
except Exception as e: print(e)
try: print('wn.lch_similarity(', syn3, syn4, '):',  wn.lch_similarity(syn3, syn4))
except Exception as e: print(e)    

print()

try: print('wn.wup_similarity(', syn1, syn1, '):',  wn.wup_similarity(syn1, syn1))
except Exception as e: print(e)
try: print('wn.wup_similarity(', syn1, syn2, '):',  wn.wup_similarity(syn1, syn2))
except Exception as e: print(e)
try: print('wn.wup_similarity(', syn1, syn3, '):',  wn.wup_similarity(syn1, syn3))
except Exception as e: print(e)
try: print('wn.wup_similarity(', syn3, syn4, '):',  wn.wup_similarity(syn3, syn4))
except Exception as e: print(e)    

print()

from nltk.corpus import wordnet_ic

ic_brown = wordnet_ic.ic('ic-brown.dat')

try: print('wn.lch_similarity(', syn1, syn1, '):',  wn.res_similarity(syn1, syn1, ic_brown))
except Exception as e: print(e)
try: print('wn.lch_similarity(', syn1, syn2, '):',  wn.res_similarity(syn1, syn2, ic_brown))
except Exception as e: print(e)
try: print('wn.lch_similarity(', syn1, syn3, '):',  wn.res_similarity(syn1, syn3, ic_brown))
except Exception as e: print(e)
try: print('wn.lch_similarity(', syn3, syn4, '):',  wn.res_similarity(syn3, syn4, ic_brown))
except Exception as e: print(e)

print()

try: print('wn.jcn_similarity(', syn1, syn1, '):',  wn.jcn_similarity(syn1, syn1, ic_brown))
except Exception as e: print(e)
try: print('wn.jcn_similarity(', syn1, syn2, '):',  wn.jcn_similarity(syn1, syn2, ic_brown))
except Exception as e: print(e)
try: print('wn.jcn_similarity(', syn1, syn3, '):',  wn.jcn_similarity(syn1, syn3, ic_brown))
except Exception as e: print(e)
try: print('wn.jcn_similarity(', syn3, syn4, '):',  wn.jcn_similarity(syn3, syn4, ic_brown))
except Exception as e: print(e)

print()

try: print('wn.lin_similarity(', syn1, syn1, '):',  wn.lin_similarity(syn1, syn1, ic_brown))
except Exception as e: print(e)
try: print('wn.lin_similarity(', syn1, syn2, '):',  wn.lin_similarity(syn1, syn2, ic_brown))
except Exception as e: print(e)
try: print('wn.lin_similarity(', syn1, syn3, '):',  wn.lin_similarity(syn1, syn3, ic_brown))
except Exception as e: print(e)
try: print('wn.lin_similarity(', syn3, syn4, '):',  wn.lin_similarity(syn3, syn4, ic_brown))
except Exception as e: print(e)

print()

# Similitud a nivel léxico - Similitud de Liu
* Como con el resto de funciones, solo consideraremos NOMBRES Y VERBOS

In [ ]:
# ================================================
# 7. SIMILITUD A NIVEL LÉXICO - SIMILITUD DE LIU
# ================================================

# 7.1 SIMILITUD SEGÚN [LIU 2013]

# 7.1.1 Espacios vectoriales de conceptos según Wordnet (para nombres y verbos)

    # Creamos espacios vectoriales para nombres y verbos a partir de las correspondientes ontologías de Wordnet
        # Cada dimensión se corresponde a un concepto de Wordnet (o lo que es lo mismo, un nodo o synset)
        # Por claridad, ordenamos los conceptos-dimensión según su posición en el árbol (ordenación "breath free")
        # Número de dimensiones:
            # Espacio de conceptos-nombre: 82.115
            # Espacio de conceptos-verbo: 13.767   
            
from nltk.corpus import wordnet as wn

def crear_espacios(categoria):
    
    if categoria == 't': espacio = [synset for synset in wn.all_synsets()]
    if categoria == 'n': espacio = [synset for synset in wn.all_synsets('n')]
    if categoria == 'v': espacio = [synset for synset in wn.all_synsets('v')]
    if categoria == 'a': espacio = [synset for synset in wn.all_synsets('a')] # incluye adjetivos a ("head") y s ("satellite").
    if categoria == 'r': espacio = [synset for synset in wn.all_synsets('r')]

    espacio = sorted(espacio, key=lambda concepto: concepto.max_depth())

    '''
    MIRAR: https://wordnet.princeton.edu/ para encontrar métodos para tratar la similitud de adjetivos y adverbios
    '''

    print('Dimensión del espacio', categoria, 'en Wordnet:', len(espacio))

    return (espacio)

# %timeit -n 1 -r 1 crear_espacios('n')
    # Respuesta: 3-4 segundos.
    # La ordenación de synsets duplica el tiempo de computación.
    
wn_todos = crear_espacios('t')
wn_nombres = crear_espacios('n')
wn_verbos = crear_espacios('v')
wn_adjetivos = crear_espacios('a')
wn_adverbios = crear_espacios('r')

print('Dimensión del espacio de nombres + verbos + adjetivos + adverbios:', len(wn_nombres) + len(wn_verbos) + len(wn_adjetivos) + len(wn_adverbios))

In [ ]:
# 7.1.2 Vector Concepto

    # Entrada: Un synset de wordnet
        # si el parámetro "info" es "print", se mostrará información durante el proceso de cálculo
    # Salida: vector concepto para ese synset

import numpy as np    

def v_concepto_liu(synset, info=''):
    
    # Inicialización del "vector concepto", que tendrá la dimensión del espacio Wordnet correspondiente

    v_concepto = np.zeros(len(wn_nombres), dtype=np.int)
    
    # Nodos relevantes para el concepto

    hyper = lambda s:s.hypernyms()
    hypo = lambda s:s.hyponyms()
    ancestros = list(synset.closure(hyper))
    descendientes = list(synset.closure(hypo))
    relevantes = [synset] + ancestros + descendientes
    relevantes =  sorted(relevantes, key=lambda s: s.max_depth())
    '''
    ELIMINAR la ordenación  si ralentiza el proceso
    '''    
    if info == 'print': 
        print('Cálculo del vector concepto de Liu, v_concepto, correspondiente a', synset)
        print()    
        print('Nodos relevantes de', synset, ':', relevantes)
        print()
        print('Valor componentes correspondientes a nodos relevantes (para nodos no relevantes, valor componente = 0):')
        print()
    
    # Peso de cada componente del vector, según si es o no nodo relevante y la densidad.
    '''
    Pendiente incluir verbos
    '''
    for index, wn_synset in enumerate(wn_nombres):
        if wn_synset in relevantes:
            if info == 'print': print('Nodo relevante:', index, wn_synset)
            if wn_synset.hypernyms() != []:
                if info == 'print': 
                    print('Calculo componente syn:', synset, '<--> wn_synset:', wn_synset)
                    print('padres de wn_synset:', wn_synset.hypernyms()[0]) # En caso de existir más de un padre, se selecciona el primero.
                    print('hermanos de wn_synset:', wn_synset.hypernyms()[0].hyponyms())
                v_concepto[index] = len(wn_synset.hypernyms()[0].hyponyms())
            v_concepto[index] = v_concepto[index] + 1  # densidad = Nº hermanos + 1
            if info == 'print': 
                print('Valor componente:', v_concepto[index])
                print()
    if info == 'print': 
        print('v_concepto para', synset, ':', v_concepto)
        print()
    return(v_concepto)


# Ejemplo para comprobar funcionamiento vector_liu()

synset1 = c['syns1'][0][0]

print('****** EJEMPLO vector_liu() ********')   
print('synset1:', synset1)
concepto_liu1 = v_concepto_liu(synset1, info='print')
print('concepto_liu1:', concepto_liu1)
print('****')
print()


In [ ]:
# 7.1.3 Similitud de Liu-Wang a nivel léxico

# función coseno
    # Entrada: dos vectores
    # Salida: Un número real entre [0,1] (coseno entre los dos vectores)

def coseno(v1, v2):
    
    coseno = v1.dot(v2) / np.sqrt(v1.dot(v1) * v2.dot(v2))
    
    return(coseno)

# Ejemplo para comprobar funcionamiento coseno()

print('****** EJEMPLO coseno() ********')   
print('Coseno:', coseno(concepto_liu1,concepto_liu1))
print('****')

In [ ]:
'''
def similitud_liu(tagged_token1, tagged_token2):
    if tagged_token1 == tagged_token2:
        sim = 1
'''
# Función similitud léxica de Liu-Wang cuando las dos palabras a comparar están en wordnet
    # Entrada: Dos synsets
    # Salida: Un número real entre [0,1] (coseno entre los dos vectores)

def similitud_lw(syn1, syn2):
    
    if syn1 == syn2:
        sim = 1
    
    elif syn1.pos() == syn2.pos() and syn1.pos() in ['n', 'v']:
        v_concepto1 = v_concepto_liu(syn1)
        v_concepto2 = v_concepto_liu(syn2)
        sim = coseno(v_concepto1, v_concepto2)
    else:
        raise Exception('Los argumentos proporcionados deben ser synsets y ambos deben ser o bien nombres o bien verbos')
    return(sim)

# Synset('son.n.01'), Synset('toy.v.02'), Synset('guitar.n.01'), Synset('black.a.02'), Synset('pawl.n.01'), Synset('issue.v.04'), Synset('water_system.n.02'), Synset('white.a.02'), Synset('testis.n.01'), Synset('mouth.n.04'), Synset('testimony.n.03'), Synset('substantial.a.03'), Synset('health.n.02'), Synset('expert.n.01')

syn1 = wn.synset('dog.n.01')
syn2 = wn.synset('cat.n.01')
syn3 = wn.synset('black.a.01')
syn4 = wn.synset('white.a.01')

try: print('similitud_lw(', syn1, syn1, '):', similitud_lw(syn1, syn1))
except Exception as e: print(e)
try: print('similitud_lw(', syn1, syn2, '):', similitud_lw(syn1, syn2))
except Exception as e: print(e)
try: print('similitud_lw(', syn1, syn3, '):', similitud_lw(syn1, syn3))
except Exception as e: print('similitud_lw(', syn1, syn3, '):', e)
try: print('similitud_lw(', syn3, syn4, '):', similitud_lw(syn3, syn4))
except Exception as e: print('similitud_lw(', syn3, syn4, '):', e)



In [ ]:
# Para palabras que no están en wordnet, definimos la similitud en función de si son idénticas o no.

def similitud_palabras(palabra1, palabra2):
    if palabra1 == palabra2:
        sim = 1
    else:
        sim = 0
    return(sim)
        
palabra1 = 'vienna'
palabra2 = 'vienna'
palabra3 = 'zaf'
similitud_palabras(palabra1, palabra2)

print ('similitud_palabras(', palabra1, ',' , palabra2, '):', similitud_palabras(palabra1, palabra2))
print ('similitud_palabras(', palabra1, ',' , palabra3, '):', similitud_palabras(palabra1, palabra3))

In [ ]:
# ========================================
# 7. SIMILITUD LÉXICA - MÉTODOS INCLUÍDOS EN WORDNET
# ========================================

import numpy as np

# 6.1 MATRIZ DE SIMILITUD 

    # función similitud similitud(syns1, syns2, medida)
        # Salida: Matriz de dimimensión len(syn1) X len(syn2)
            # Cada valor [i, j] es la similitud entre el elemento i de syns1 y el elemento j de syns2
            # Si no existe similitud, p. ej por tener PoS distintos, la función mantiene el valor cero asignado en la inicialización
        # Argumentos:
            # syns1, syns2: listas de synsets a comparar
            # medida: función wordnet de estimación de la similitud léxica:
                # ps --> path_similarity (POR DEFECTO). Rango: [0:1]
''' 
            MIRAR:  By default, there is now a fake root node added to verbs so for cases 
            where previously a path could not be found---and None was returned---it should return a value. 
            The old behavior can be achieved by setting simulate_root to be False.
'''
            # lch --> Leacock Chodorow. Rango: [0:3.64?] 
'''
                MIRAR upper bound per normalitzar: https://stackoverflow.com/questions/20112828/maximum-score-in-wordnet-based-similarity
                >>> max(max(len(hyp_path) for hyp_path in ss.hypernym_paths()) for ss in wn.all_synsets())

'''
            # wup --> Wu_Palmer
'''
                MIRAR: range
'''
            # res --> Resknik
        # ic: el corpus, en formato de "Information Content", para aquellas funciones de similitud que lo requieren

from nltk.corpus import wordnet_ic

def similitud(syns1, syns2, medida='ps', ic = ''):
    s = np.zeros(shape = (len(syns1), len(syns2)))    # similitud entre los syns
    # s_errors = np.zeros(shape = (len(errors1), len(errors2)))    # similitud entre los errores    
    for i in range(len(syns1)):
        for j in range(len(syns2)):
            if (syns1[i].pos() == syns2[j].pos()):
                if medida == 'ps':
                    sim = wn.path_similarity(syns1[i], syns2[j])
                elif medida == 'lch':
                    sim = wn.lch_similarity(syns1[i], syns2[j])
                elif medida == 'wup':
                    sim = wn.wup_similarity(syns1[i], syns2[j])
                elif medida == 'res':
                    sim = wn.res_similarity(syns1[i], syns2[j], ic)
                elif medida == 'jcn':
                    sim = wn.jcn_similarity(syns1[i], syns2[j], ic)
                elif medida == 'lin':
                    sim = wn.lin_similarity(syns1[i], syns2[j], ic)
                if sim is None:
                    continue
                else:
                    s[i,j] = sim
    return(s)


# Ejemplo para comprobar funcionamiento similitud()

syns1 = c['syns1'][501]
syns2 = c['syns2'][501]
ic_brown = wordnet_ic.ic('ic-brown.dat')

print('****** EJEMPLO similitud() ********')   
print('syns1:', syns1)
print('syns2:', syns2)
print('similitud(syns1, syns2, \'ps\'):')
print(similitud(syns1, syns2, 'ps'))
print('similitud(syns1, syns2, \'lch\'):')
print(similitud(syns1, syns2, 'lch'))
print('similitud(syns1, syns2, \'wup\'):')
print(similitud(syns1, syns2, 'wup'))
print('similitud(syns1, syns2, \'res\', ic_brown):')
print(similitud(syns1, syns2, 'res', ic_brown))
print('similitud(syns1, syns2, \'jcn\', ic_brown):')
print(similitud(syns1, syns2, 'jcn', ic_brown))
print('similitud(syns1, syns2, \'lin\', ic_brown):')
print(similitud(syns1, syns2, 'lin', ic_brown))
print('****')


In [ ]:
# Creación y almacenamiento de las matrices de similitud

'''
    Para el análisis final con todos los datos, no almacenar, para optimizar memoria.
'''

c['matriz_sim_ps'] = ''
c['matriz_sim_lch'] = ''
c['matriz_sim_wup'] = ''
c['matriz_sim_res_brown'] = ''
c['matriz_sim_jcn_brown'] = ''
c['matriz_sim_lin_brown'] = ''

for i in c.index:
    syns1 = c.at[i, 'syns1']
    syns2 = c.at[i, 'syns2']
    c.at[i, 'matriz_sim_ps'] = similitud(syns1, syns2, 'ps')
    c.at[i, 'matriz_sim_lch'] = similitud(syns1, syns2, 'lch')
    c.at[i, 'matriz_sim_wup'] = similitud(syns1, syns2, 'wup')

ic_brown = wordnet_ic.ic('ic-brown.dat')

for i in c.index:    
    c.at[i, 'matriz_sim_res_brown'] = similitud(syns1, syns2, 'res', ic_brown)    
    c.at[i, 'matriz_sim_jcn_brown'] = similitud(syns1, syns2, 'jcn', ic_brown)    
    c.at[i, 'matriz_sim_lin_brown'] = similitud(syns1, syns2, 'lin', ic_brown)
    
c.head()

In [ ]:
# 6.2 PUNTUACIÓN TOTAL

    # Función puntuación
        # PASO 1: Vector puntuaciones: vector de dimensión el número de filas de matriz_sim (o sea, número de elementos de syns1)
            # En cada componente se escribe la similitud máxima el elemento de syns1 y los elementos del syns2.
        # PASO 2: Puntuacion: media de todos los scores.

def puntuacion(matriz_sim):
    
    puntuaciones = matriz_sim.max(axis=1)
    print('PUNTUACIONES:', puntuaciones)
       
    puntuacion = puntuaciones.mean()
    print('MEDIA:', puntuacion)
    
    return(puntuacion)

# Ejemplo para comprobar funcionamiento puntuacion()

matriz_sim = c['matriz_sim_ps'][0]

print('****** EJEMPLO puntuación() ********')   
print('matriz_sim:')
print(matriz_sim)
print('puntuacion(matriz_sim):', puntuacion(matriz_sim))
print('****')
print()



In [ ]:
# Cálculo y almacenamiento de la puntuación

c['puntuacion_ps'] = 0.0
c['puntuacion_lch'] = 0.0
c['puntuacion_wup'] = 0.0
c['puntuacion_res_brown'] = 0.0
c['puntuacion_jcn_brown'] = 0.0
c['puntuacion_lin_brown'] = 0.0

for i in c.index:
    print('FILA:', i)
    print('Path_similarity')
    if c.at[i, 'matriz_sim_ps'].size != 0:
        c.at[i, 'puntuacion_ps'] = puntuacion(c.at[i, 'matriz_sim_ps'])
    print('Leacock-Chodorow')
    if c.at[i, 'matriz_sim_lch'].size != 0:
        c.at[i, 'puntuacion_lch'] = puntuacion(c.at[i, 'matriz_sim_lch'])
    print('Wu-Palmer')
    if c.at[i, 'matriz_sim_wup'].size != 0:
        c.at[i, 'puntuacion_wup'] = puntuacion(c.at[i, 'matriz_sim_wup'])
    print('Resnik (Brown)')
    if c.at[i, 'matriz_sim_res_brown'].size != 0:
        c.at[i, 'puntuacion_res_brown'] = puntuacion(c.at[i, 'matriz_sim_res_brown'])
    print('Jiang-Conrath (Brown)')
    if c.at[i, 'matriz_sim_jcn_brown'].size != 0:
        c.at[i, 'puntuacion_jcn_brown'] = puntuacion(c.at[i, 'matriz_sim_jcn_brown'])
    print('Lin (Brown)')
    if c.at[i, 'matriz_sim_lin_brown'].size != 0:
        c.at[i, 'puntuacion_lin_brown'] = puntuacion(c.at[i, 'matriz_sim_lin_brown'])
    print()
    

In [ ]:
c

In [ ]:
c.iloc[:, 7:12].head()

In [ ]:
# ========================================
# 8. SIMILITUD DE LIU A NIVEL DE FRASE - MÉTODO LIU-WANG
# ========================================

# 8.1 Vector semántico
    # Entrada: Un conjunto de synsets (frase) y una bag of words (bow) conteniendo el conjunto de synsets
        # si el parámetro "info" es "print", se mostrará información durante el proceso de cálculo
    # Salida: Vector semántico del conjunto de synsets en el espacio definido por bow

def v_semantico0(syns, bow, info=''):

    v = np.zeros(len(bow), dtype=np.float)

    if info == 'print': 
        print('bow:', bow)
        print('syns:', syns)
        print()
        print('Cálculo del vector semántico v correspondiente a syns en el espacio bow')
        print('v:', v)
        print()    

    for index, synset in enumerate(bow):  
        if info == 'print': 
            print('Componente de v (index):', index)
            print('synset de bow correspondiente a index =',index, ':', synset)
            print('Valor inicial de la componente v', index, ':', v[index])
            print()
            print('Calculo del valor de la componente v', index)
            print()
        sims = list()
        for syn in syns:
            if info == 'print':
                print('Cálculo de la similitud de synset de bow con synset de syns:', synset, '<-->', syn)
                print()
            if syn.pos() == synset.pos():
                sim = similitud_lw(syn, synset)
                if info == 'print': print(sim)
                '''
                Estamos repitiendo el mismo cálculo iteración tras iteración
                MODIFICAR: Calcular todos los vectores concepto del dataset al principio
                '''
            else:
                sim = 0.0
                if info == 'print':
                    print(synset, 'y', syn, 'no pertenecen a la misma categoría gramatical')    
                    print('Similitud Liu ', synset, '<-->', syn, ':', sim)
            if sim != 'nan': 
                sims.append(sim)
            if info == 'print': print()
        v[index] = max(sims)
        if info == 'print':
            print('Valores de las similitudes:', sims)
            print("Valor de la componente v", index, '(máximo de las similitudes):', v[index])
            print()
    if info == 'print': print('v = ', v)
    if info == 'print': print()
    return(v)
    
# Ejemplo para comprobar funcionamiento v_semantico_liu()

print('****** EJEMPLO v_semantico() ********')   
bow = c['bow'][0]
syns1 = c['syns1'][0]
syns2 = c['syns2'][0]
v1 = v_semantico0(syns1, bow, info='')
v2 = v_semantico0(syns2, bow, info='')
print('bow:', bow)
print('syns1:', syns1)
print('Vector semántico:', v1)
print('syns2:', syns2)
print('Vector semántico:', v2)
print('****')

In [ ]:
# ========================================
# 8. SIMILITUD A NIVEL DE FRASE - LIU-WANG
# ========================================

# 8.1 Vector semántico
    # Entrada: Un conjunto de synsets (frase) y una bag of words (bow) conteniendo el conjunto de synsets
        # si el parámetro "info" es "print", se mostrará información durante el proceso de cálculo
    # Argumentos: 
        # medida : función de similitud léxica que se quiere usar:
            # ps --> path_similarity (POR DEFECTO). Rango: [0:1]
            # lch --> Leacock Chodorow. Rango: [0:3.64?] 
            # wup --> Wu_Palmer
            # res --> Resknik (requiere corpus ic)
            # jcn --> Jiang-Conrath (requiere corpus ic)
            # lin --> Lin (requiere corpus ic)
            # lw --> Liu-Wang
        # ic: corpus ic, para las funciones que lo requieran.
            # Intentar reproducir https://www.kdnuggets.com/2017/11/building-wikipedia-text-corpus-nlp.html
    # Salida: Vector semántico del conjunto de synsets en el espacio definido por bow

def v_semantico(syns, bow, medida='ps', ic = '', info=''):
    v = np.zeros(len(bow), dtype=np.float)    

    if info == 'print': 
        print('bow:', bow)
        print('syns:', syns)
        print()
        print('Cálculo del vector semántico v correspondiente a syns en el espacio bow')
        print('v:', v)
        print()    
        print('Método de similitud léxica:', medida)
        if (medida == 'res' or medida == 'jcn' or medida == 'lin'): 
            print('Corpus (Information content:)', ic)
        print()       

    for index, synset in enumerate(bow):  
        if info == 'print': 
            print('Componente de v (index):', index)
            print('synset de bow correspondiente a index =',index, ':', synset)
            print('Valor inicial de la componente v', index, ':', v[index])
            print()
            print('Calculo del valor de la componente v', index)
            print()
        sims = list()
        for syn in syns:
            if info == 'print':
                print('Cálculo de la similitud de synset de bow con synset de syns:', synset, '<-->', syn)
                print()
            sim = 0.0
            if syn.pos() == synset.pos():
                if medida == 'ps':
                    sim = wn.path_similarity(syn, synset)
                elif medida == 'lch':
                    sim = wn.lch_similarity(syn, synset)
                elif medida == 'wup':
                    sim = wn.wup_similarity(syn, synset)
                elif medida == 'res':
                    from nltk.corpus import wordnet_ic
                    if ic == '': ic = wordnet_ic.ic('ic-brown.dat')
                    sim = wn.res_similarity(syn, synset, ic)
                elif medida == 'jcn':
                    from nltk.corpus import wordnet_ic
                    if ic == '': ic = wordnet_ic.ic('ic-brown.dat')
                    sim = wn.jcn_similarity(syn, synset, ic)
                elif medida == 'lin':
                    from nltk.corpus import wordnet_ic
                    if ic == '': ic = wordnet_ic.ic('ic-brown.dat')
                    sim = wn.lin_similarity(syn, synset, ic)
                elif medida == 'lw':
                    sim = similitud_lw(syn, synset)    #Estamos repitiendo el mismo cálculo iteración tras iteración MODIFICAR: Calcular todos los vectores concepto del dataset al principio        
                if info == 'print':
                    print('similitud:', sim)
                if sim is None: continue
                    
            else:
                sim = 0.0
                if info == 'print':
                    print(synset, 'y', syn, 'no pertenecen a la misma categoría gramatical')    
                    print('Similitud Liu ', synset, '<-->', syn, ':', sim)
            if sim != 'nan': 
                sims.append(sim)
            if info == 'print': print()
        v[index] = max(sims)
        if info == 'print':
            print('Valores de las similitudes:', sims)
            print("Valor de la componente v", index, '(máximo de las similitudes):', v[index])
            print()
    if info == 'print': print('v = ', v)
    if info == 'print': print()
    return(v)

    
# Ejemplo para comprobar funcionamiento v_semantico()

print('****** EJEMPLO v_semantico() ********')   

bow = c['bow'][101]
syns1 = c['syns1'][101]
syns2 = c['syns2'][101]
info = ''
ic = ''

print('bow:', bow)
print('syns1:', syns1)
print('syns2:', syns2)

print()
print('SIMILITUD LIU - WANG')

medida = 'lw'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

print()
print('SIMILITUDES DE WN QUE NO REQUIEREN IC')

medida = 'ps'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'lch'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'wup'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

print()
print('SIMILITUDES DE WN QUE REQUIEREN IC. IC = BROWN')

ic = ''

medida = 'res'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'jcn'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'lin'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info='print'))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'\'):', v_semantico(syns2, bow,medida=medida, ic=ic, info='print'))

print()
print('SIMILITUDES DE WN QUE REQUIEREN IC. IC = SEMCOR')
     
from nltk.corpus import wordnet_ic
ic_brown = wordnet_ic.ic('ic-brown.dat')
ic = ic_brown

medida = 'res'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'jcn'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'lin'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

print()
print('SIMILITUDES DE WN QUE REQUIEREN IC. IC = GENESIS')
     
from nltk.corpus import genesis
ic_genesis = wn.ic(genesis, False, 0.0)
ic = ic_genesis

medida = 'res'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'jcn'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))

medida = 'lin'
print('v_semantico(syns1, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns1, bow, medida=medida, ic=ic, info=info))
print('v_semantico(syns2, bow, medida=\'', medida, '\', info=\'',  info, '\' , ic=\'semcor\'):', v_semantico(syns2, bow, medida=medida, ic=ic, info=info))


print('****')


In [ ]:
# 8.2 Similitud de frase Liu-Wang
    # Entrada: Dos vectores semánticos
    # Salida: Similitud entre las dos frases, calculada como el coseno entre los dos vectores semánticos
    
def similitud_frase_lw(v1, v2):
        
    sim = coseno(v1, v2)
    
    return(sim)

# Ejemplo para comprobar funcionamiento similitud_frase_liu()

print('****** EJEMPLO similitud_frase() ********')   

v1 = v_semantico(syns1, bow, medida=medida, ic=ic, info=info)
v2 = v_semantico(syns2, bow, medida=medida, ic=ic, info=info)
print('Similitud a nivel de frase', similitud_frase_lw(v1, v2))
print('****')

In [ ]:
# Cálculo y almacenamiento de la puntuación Liu-Wang a nivel de frase - 1

c['v1_lw'] = ''
c['v2_lw'] = ''
c['puntuacion_lw_lw'] = 0.0
c['v1_ps'] = ''  # OK
c['v2_ps'] = ''  # OK
c['puntuacion_lw_ps'] = 0.0  # OK, Pearson 0.66
c['v1_lch'] = ''  # OK
c['v2_lch'] = ''  # OK
c['puntuacion_lw_lch'] = 0.0  # OK, Pearson 0.49
c['v1_wup'] = '' # OK
c['v2_wup'] = '' # OK
c['puntuacion_lw_wup'] = 0.0 # OK, Pearson 0.42
c['v1_res_brown'] = '' # Resultats no fiables quan la paraula no està al Corpus --> PROVAR IF ABANS; BoW_nv!!!!
c['v2_res_brown'] = ''  # Resultats no fiables quan la paraula no està al Corpus --> PROVAR IF ABANS; BoW_nv!!!!
c['puntuacion_lw_res_brown'] = 0.0 # OK, Pearson 0.77 MILLORABLE 
c['v1_jcn_brown'] = '' # Components infinites quan les paraules són iguals --> ESTABLIR UN MÀXIM ; BoW_nv!!!!
c['v2_jcn_brown'] = '' # Components infinites quan les paraules són iguals --> ESTABLIR UN MÀXIM ; BoW_nv!!!!
c['puntuacion_lw_jcn_brown'] = 0.0  # Tots els resultats són zero.
c['v1_lin_brown'] = ''
c['v2_lin_brown'] = ''
c['puntuacion_lw_lin_brown'] = 0.0
c['v1_res_semcor'] = ''
c['v2_res_semcor'] = ''
c['puntuacion_lw_res_semcor'] = 0.0
c['v1_jcn_semcor'] = ''
c['v2_jcn_semcor'] = ''
c['puntuacion_lw_jcn_semcor'] = 0.0
c['v1_lin_semcor'] = ''
c['v2_lin_semcor'] = ''
c['puntuacion_lw_lin_semcor'] = 0.0
c['v1_res_genesis'] = ''
c['v2_res_genesis'] = ''
c['puntuacion_lw_res_genesis'] = 0.0
c['v1_jcn_genesis'] = ''
c['v2_jcn_genesis'] = ''
c['puntuacion_lw_jcn_genesis'] = 0.0
c['v1_lin_genesis'] = ''
c['v2_lin_genesis'] = ''
c['puntuacion_lw_lin_genesis'] = 0.0

print_info = input("Mostrar los detalles del procesamiento? (s/n)")

from nltk.corpus import wordnet_ic
ic_semcor = wordnet_ic.ic('ic-semcor.dat')
from nltk.corpus import genesis
ic_genesis = wn.ic(genesis, False, 0.0)

for i in c.index:
    start = time.time()
    if print_info == 's':
        print('FILA:', i)
        print('Gold score:', c.at[i, 'gold_score'])
        print('sent1:', c.at[i, 'sent1'])
        print('sen  t2:', c.at[i, 'sent2'])
        print('syns1:', c.at[i, 'syns1'])
        print('syns2:', c.at[i, 'syns2'])
        print('bow:', c.at[i, 'bow'])
    try: c.at[i, 'v1_lw'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v1_lw'] = None
    try: c.at[i, 'v2_lw'] = v_semantico(c.at[i, 'syns2'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v2_lw'] = None
    try: c.at[i, 'puntuacion_lw_lw'] = similitud_frase_lw(c.at[i, 'v1_lw'], c.at[i, 'v2_lw'] )
    except:  c.at[i, 'puntuacion_lw_lw'] = None
        
    if print_info == 's':
        print('v1_lw', i, ': ', c.at[i, 'v1_lw'])
        print('v2_lw', i, ': ', c.at[i, 'v2_lw'])
        print('puntuacion_lw_lw', i, ': ', c.at[i, 'puntuacion_lw_lw'])

    try: c.at[i, 'v1_ps'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='ps', ic='')
    except:  c.at[i, 'v1_ps'] = None
    try: c.at[i, 'v2_ps'] = v_semantico(c.at[i, 'syns2'], c.at[i, 'bow'], medida='ps', ic='')
    except:  c.at[i, 'v2_ps'] = None
    try: c.at[i, 'puntuacion_lw_ps'] = similitud_frase_lw(c.at[i, 'v1_ps'], c.at[i, 'v2_ps'] )
    except:  c.at[i, 'puntuacion_lw_ps'] = None

    if print_info == 's':
        print('v1_ps', i, ': ', c.at[i, 'v1_ps'])
        print('v2_ps', i, ': ', c.at[i, 'v2_ps'])
        print('puntuacion_lw_ps', i, ': ', c.at[i, 'puntuacion_lw_ps'])

    try: c.at[i, 'v1_lch'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lch', ic='')
    except:  c.at[i, 'v1_lch'] = None
    try: c.at[i, 'v2_lch'] = v_semantico(c.at[i, 'syns2'], c.at[i, 'bow'], medida='lch', ic='')
    except:  c.at[i, 'v2_lch'] = None
    try: c.at[i, 'puntuacion_lw_lch'] = similitud_frase_lw(c.at[i, 'v1_lch'], c.at[i, 'v2_lch'] )
    except:  c.at[i, 'puntuacion_lw_lch'] = None

    if print_info == 's':
        print('v1_lch', i, ': ', c.at[i, 'v1_lch'])
        print('v2_lch', i, ': ', c.at[i, 'v2_lch'])
        print('puntuacion_lw_lch', i, ': ', c.at[i, 'puntuacion_lw_lch'])

    try: c.at[i, 'v1_wup'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='wup', ic='')
    except:  c.at[i, 'v1_wup'] = None
    try: c.at[i, 'v2_wup'] = v_semantico(c.at[i, 'syns2'], c.at[i, 'bow'], medida='wup', ic='')
    except:  c.at[i, 'v2_wup'] = None
    try: c.at[i, 'puntuacion_lw_wup'] = similitud_frase_lw(c.at[i, 'v1_wup'], c.at[i, 'v2_wup'] )
    except:  c.at[i, 'puntuacion_lw_wup'] = None

    if print_info == 's':
        print('v1_wup', i, ': ', c.at[i, 'v1_wup'])
        print('v2_wup', i, ': ', c.at[i, 'v2_wup'])
        print('puntuacion_lw_wup', i, ': ', c.at[i, 'puntuacion_lw_wup'])

    try: c.at[i, 'v1_res_brown'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='res', ic='')
    except:  c.at[i, 'v1_res_brown'] = None
    try: c.at[i, 'v2_res_brown'] = v_semantico(c.at[i, 'syns2'], c.at[i, 'bow'], medida='res', ic='')
    except:  c.at[i, 'v2_res_brown'] = None
    try: c.at[i, 'puntuacion_lw_res_brown'] = similitud_frase_lw(c.at[i, 'v1_res_brown'], c.at[i, 'v2_res_brown'] )
    except:  c.at[i, 'puntuacion_lw_res_brown'] = None

    if print_info == 's':
        print('v1_res_brown', i, ': ', c.at[i, 'v1_res_brown'])
        print('v2_res_brown', i, ': ', c.at[i, 'v2_res_brown'])
        print('puntuacion_lw_res_brown', i, ': ', c.at[i, 'puntuacion_lw_res_brown'])

    from nltk.corpus import wordnet_ic
    ic_semcor = wordnet_ic.ic('ic-semcor.dat')
    ic = ic_semcor
    
    try: c.at[i, 'v1_res_semcor'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='res', ic=ic_semcor)
    except:  c.at[i, 'v1_res_semcor'] = None
    try: c.at[i, 'v2_res_semcor'] = v_semantico(c.at[i, 'syns2'], c.at[i, 'bow'], medida='res', ic=ic_semcor)
    except:  c.at[i, 'v2_res_semcor'] = None
    try: c.at[i, 'puntuacion_lw_res_semcor'] = similitud_frase_lw(c.at[i, 'v1_res_semcor'], c.at[i, 'v2_res_semcor'] )
    except:  c.at[i, 'puntuacion_lw_res_semcor'] = None

    if print_info == 's':
        print('v1_res_semcor', i, ': ', c.at[i, 'v1_res_semcor'])
        print('v2_res_semcor', i, ': ', c.at[i, 'v2_res_semcor'])
        print('puntuacion_lw_res_semcor', i, ': ', c.at[i, 'puntuacion_lw_res_semcor'])

    from nltk.corpus import genesis
    ic_genesis = wn.ic(genesis, False, 0.0)
    ic = ic_genesis
    
    try: c.at[i, 'v1_res_genesis'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='res', ic=ic_genesis)
    except:  c.at[i, 'v1_res_genesis'] = None
    try: c.at[i, 'v2_res_genesis'] = v_semantico(c.at[i, 'syns2'], c.at[i, 'bow'], medida='res', ic=ic_genesis)
    except:  c.at[i, 'v2_res_genesis'] = None
    try: c.at[i, 'puntuacion_lw_res_genesis'] = similitud_frase_lw(c.at[i, 'v1_res_genesis'], c.at[i, 'v2_res_genesis'] )
    except:  c.at[i, 'puntuacion_lw_res_genesis'] = None

    if print_info == 's':
        print('v1_res_genesis', i, ': ', c.at[i, 'v1_res_genesis'])
        print('v2_res_genesis', i, ': ', c.at[i, 'v2_res_genesis'])
        print('puntuacion_lw_res_genesis', i, ': ', c.at[i, 'puntuacion_lw_res_genesis'])      
        
    try: c.at[i, 'v1_jcn_brown'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='jcn', ic='')
    except:  c.at[i, 'v1_jcn_brown'] = None
    try: c.at[i, 'v2_jcn_brown'] = v_semantico(c.at[i, 'syns2'], c.at[i, 'bow'], medida='jcn', ic='')
    except:  c.at[i, 'v2_jcn_brown'] = None
    try: c.at[i, 'puntuacion_lw_jcn_brown'] = similitud_frase_lw(c.at[i, 'v1_jcn_brown'], c.at[i, 'v2_jcn_brown'] )
    except:  c.at[i, 'puntuacion_lw_jcn_brown'] = None

    if print_info == 's':
        print('v1_jcn_brown', i, ': ', c.at[i, 'v1_jcn_brown'])
        print('v2_jcn_brown', i, ': ', c.at[i, 'v2_jcn_brown'])
        print('puntuacion_lw_jcn_brown', i, ': ', c.at[i, 'puntuacion_lw_jcn_brown'])

    from nltk.corpus import wordnet_ic
    ic_semcor = wordnet_ic.ic('ic-semcor.dat')
    ic = ic_semcor
    
    try: c.at[i, 'v1_jcn_semcor'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='jcn', ic=ic_semcor)
    except:  c.at[i, 'v1_jcn_semcor'] = None
    try: c.at[i, 'v2_jcn_semcor'] = v_semantico(c.at[i, 'syns2'], c.at[i, 'bow'], medida='jcn', ic=ic_semcor)
    except:  c.at[i, 'v2_jcn_semcor'] = None
    try: c.at[i, 'puntuacion_lw_jcn_semcor'] = similitud_frase_lw(c.at[i, 'v1_jcn_semcor'], c.at[i, 'v2_jcn_semcor'] )
    except:  c.at[i, 'puntuacion_lw_jcn_semcor'] = None

    if print_info == 's':
        print('v1_jcn_semcor', i, ': ', c.at[i, 'v1_jcn_semcor'])
        print('v2_jcn_semcor', i, ': ', c.at[i, 'v2_jcn_semcor'])
        print('puntuacion_lw_jcn_semcor', i, ': ', c.at[i, 'puntuacion_lw_jcn_semcor'])

    from nltk.corpus import genesis
    ic_genesis = wn.ic(genesis, False, 0.0)
    ic = ic_genesis
    
    try: c.at[i, 'v1_jcn_genesis'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='jcn', ic=ic_genesis)
    except:  c.at[i, 'v1_jcn_genesis'] = None
    try: c.at[i, 'v2_jcn_genesis'] = v_semantico(c.at[i, 'syns2'], c.at[i, 'bow'], medida='jcn', ic=ic_genesis)
    except:  c.at[i, 'v2_jcn_genesis'] = None
    try: c.at[i, 'puntuacion_lw_jcn_genesis'] = similitud_frase_lw(c.at[i, 'v1_jcn_genesis'], c.at[i, 'v2_jcn_genesis'] )
    except:  c.at[i, 'puntuacion_lw_jcn_genesis'] = None

    if print_info == 's':
        print('v1_jcn_genesis', i, ': ', c.at[i, 'v1_jcn_genesis'])
        print('v2_jcn_genesis', i, ': ', c.at[i, 'v2_jcn_genesis'])
        print('puntuacion_lw_jcn_genesis', i, ': ', c.at[i, 'puntuacion_lw_jcn_genesis'])

    try: c.at[i, 'v1_lin_brown'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lin', ic='')
    except:  c.at[i, 'v1_lin_brown'] = None
    try: c.at[i, 'v2_lin_brown'] = v_semantico(c.at[i, 'syns2'], c.at[i, 'bow'], medida='lin', ic='')
    except:  c.at[i, 'v2_lin_brown'] = None
    try: c.at[i, 'puntuacion_lw_lin_brown'] = similitud_frase_lw(c.at[i, 'v1_lin_brown'], c.at[i, 'v2_lin_brown'] )
    except:  c.at[i, 'puntuacion_lw_lin_brown'] = None

    if print_info == 's':
        print('v1_lin_brown', i, ': ', c.at[i, 'v1_lin_brown'])
        print('v2_lin_brown', i, ': ', c.at[i, 'v2_lin_brown'])
        print('puntuacion_lw_lin_brown', i, ': ', c.at[i, 'puntuacion_lw_lin_brown'])

    from nltk.corpus import wordnet_ic
    ic_semcor = wordnet_ic.ic('ic-semcor.dat')
    ic = ic_semcor
    
    try: c.at[i, 'v1_lin_semcor'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lin', ic=ic_semcor)
    except:  c.at[i, 'v1_lin_semcor'] = None
    try: c.at[i, 'v2_lin_semcor'] = v_semantico(c.at[i, 'syns2'], c.at[i, 'bow'], medida='lin', ic=ic_semcor)
    except:  c.at[i, 'v2_lin_semcor'] = None
    try: c.at[i, 'puntuacion_lw_lin_semcor'] = similitud_frase_lw(c.at[i, 'v1_lin_semcor'], c.at[i, 'v2_lin_semcor'] )
    except:  c.at[i, 'puntuacion_lw_lin_semcor'] = None

    if print_info == 's':
        print('v1_lin_semcor', i, ': ', c.at[i, 'v1_lin_semcor'])
        print('v2_lin_semcor', i, ': ', c.at[i, 'v2_lin_semcor'])
        print('puntuacion_lw_lin_semcor', i, ': ', c.at[i, 'puntuacion_lw_lin_semcor'])

    from nltk.corpus import genesis
    ic_genesis = wn.ic(genesis, False, 0.0)
    ic = ic_genesis
    
    try: c.at[i, 'v1_lin_genesis'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lin', ic=ic_genesis)
    except:  c.at[i, 'v1_lin_genesis'] = None
    try: c.at[i, 'v2_lin_genesis'] = v_semantico(c.at[i, 'syns2'], c.at[i, 'bow'], medida='lin', ic=ic_genesis)
    except:  c.at[i, 'v2_lin_genesis'] = None
    try: c.at[i, 'puntuacion_lw_lin_genesis'] = similitud_frase_lw(c.at[i, 'v1_lin_genesis'], c.at[i, 'v2_lin_genesis'] )
    except:  c.at[i, 'puntuacion_lw_lin_genesis'] = None

    if print_info == 's':
        print('v1_lin_genesis', i, ': ', c.at[i, 'v1_lin_genesis'])
        print('v2_lin_genesis', i, ': ', c.at[i, 'v2_lin_genesis'])
        print('puntuacion_lw_lin_genesis', i, ': ', c.at[i, 'puntuacion_lw_lin_genesis'])    
        
    end = time.time()
    print(end-start)


In [ ]:
c.to_excel('output.xlsx')

In [ ]:
help(c.to_excel('c.xlsx'))

In [ ]:
# Cálculo y almacenamiento de la puntuación Liu-Wang a nivel de frase - 2
'''

c['v1_lw'] = ''
c['v2_lw'] = ''
c['puntuacion_lw_lw'] = 0.0
c['v1_ps'] = ''
c['v2_ps'] = ''
c['puntuacion_lw_ps'] = 0.0
c['v1_lch'] = ''
c['v2_lch'] = ''
c['puntuacion_lw_lch'] = 0.0
c['v1_wup'] = ''
c['v2_wup'] = ''
c['puntuacion_lw_wup'] = 0.0
c['v1_res_brown'] = ''
c['v2_res_brown'] = ''
c['puntuacion_lw_res_brown'] = 0.0
c['v1_jcn_brown'] = ''
c['v2_jcn_brown'] = ''
c['puntuacion_lw_jcn_brown'] = 0.0
c['v1_lin_brown'] = ''
c['v2_lin_brown'] = ''
c['puntuacion_lw_lin_brown'] = 0.0
c['v1_res_semcor'] = ''
c['v2_res_semcor'] = ''
c['puntuacion_lw_res_semcor'] = 0.0
c['v1_jcn_semcor'] = ''
c['v2_jcn_semcor'] = ''
c['puntuacion_lw_jcn_semcor'] = 0.0
c['v1_lin_semcor'] = ''
c['v2_lin_semcor'] = ''
c['puntuacion_lw_lin_semcor'] = 0.0
c['v1_res_genesis'] = ''
c['v2_res_genesis'] = ''
c['puntuacion_lw_res_genesis'] = 0.0
c['v1_jcn_genesis'] = ''
c['v2_jcn_genesis'] = ''
c['puntuacion_lw_jcn_genesis'] = 0.0
c['v1_lin_genesis'] = ''
c['v2_lin_genesis'] = ''
c['puntuacion_lw_lin_genesis'] = 0.0

print_info = input("Would you like to print computing info on screen (y/n)")

for i in c.index:
    if print_info == 'y':
        print('FILA:', i)
        print('sent1:', c.at[i, 'sent1'])
        print('sent2:', c.at[i, 'sent2'])
        print('syns1:', c.at[i, 'syns1'])
        print('syns2:', c.at[i, 'syns2'])
        print('bow:', c.at[i, 'bow'])
    try: c.at[i, 'v1_lw'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v1_lw'] = 'UNK'
    try: c.at[i, 'v2_lw'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v2_lw'] = 'UNK'
    try: c.at[i, 'puntuacion_lw_lw'] = similitud_frase_lw(c.at[i, 'v1_lw'], c.at[i, 'v2_lw'] )
    except:  c.at[i, 'puntuacion_lw_lw'] = 'UNK'
    try: c.at[i, 'v1_ps'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v1_ps'] = 'UNK'
    try: c.at[i, 'v2_ps'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v2_ps'] = 'UNK'
    try: c.at[i, 'puntuacion_lw_ps'] = similitud_frase_lw(c.at[i, 'v1_ps'], c.at[i, 'v2_ps'] )
    except:  c.at[i, 'puntuacion_lw_ps'] = 'UNK'
    try: c.at[i, 'v1_lch'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v1_lch'] = 'UNK'
    try: c.at[i, 'v2_lch'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v2_lch'] = 'UNK'
    try: c.at[i, 'puntuacion_lw_lch'] = similitud_frase_lw(c.at[i, 'v1_lch'], c.at[i, 'v2_lch'] )
    except:  c.at[i, 'puntuacion_lw_lch'] = 'UNK'
    try: c.at[i, 'v1_wup'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v1_wup'] = 'UNK'
    try: c.at[i, 'v2_wup'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v2_wup'] = 'UNK'
    try: c.at[i, 'puntuacion_lw_wup'] = similitud_frase_lw(c.at[i, 'v1_wup'], c.at[i, 'v2_wup'] )
    except:  c.at[i, 'puntuacion_lw_wup'] = 'UNK'
    try: c.at[i, 'v1_res_brown'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v1_res_brown'] = 'UNK'
    try: c.at[i, 'v2_res_brown'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v2_res_brown'] = 'UNK'
    try: c.at[i, 'puntuacion_lw_res_brown'] = similitud_frase_lw(c.at[i, 'v1_res_brown'], c.at[i, 'v2_res_brown'] )
    except:  c.at[i, 'puntuacion_lw_res_brown'] = 'UNK'
    try: c.at[i, 'v1_jcn_brown'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v1_jcn_brown'] = 'UNK'
    try: c.at[i, 'v2_jcn_brown'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v2_jcn_brown'] = 'UNK'
    try: c.at[i, 'puntuacion_lw_jcn_brown'] = similitud_frase_lw(c.at[i, 'v1_jcn_brown'], c.at[i, 'v2_jcn_brown'] )
    except:  c.at[i, 'puntuacion_lw_jcn_brown'] = 'UNK'
    try: c.at[i, 'v1_lin_brown'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v1_lin_brown'] = 'UNK'
    try: c.at[i, 'v2_lin_brown'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v2_lin_brown'] = 'UNK'
    try: c.at[i, 'puntuacion_lw_lin_brown'] = similitud_frase_lw(c.at[i, 'v1_lin_brown'], c.at[i, 'v2_lin_brown'] )
    except:  c.at[i, 'puntuacion_lw_lin_brown'] = 'UNK'
    try: c.at[i, 'v1_res_semcor'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v1_res_semcor'] = 'UNK'
    try: c.at[i, 'v2_res_semcor'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v2_res_semcor'] = 'UNK'
    try: c.at[i, 'puntuacion_lw_res_semcor'] = similitud_frase_lw(c.at[i, 'v1_res_semcor'], c.at[i, 'v2_res_semcor'] )
    except:  c.at[i, 'puntuacion_lw_res_semcor'] = 'UNK'
    try: c.at[i, 'v1_jcn_semcor'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v1_jcn_semcor'] = 'UNK'
    try: c.at[i, 'v2_jcn_semcor'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v2_jcn_semcor'] = 'UNK'
    try: c.at[i, 'puntuacion_lw_jcn_semcor'] = similitud_frase_lw(c.at[i, 'v1_jcn_semcor'], c.at[i, 'v2_jcn_semcor'] )
    except:  c.at[i, 'puntuacion_lw_jcn_semcor'] = 'UNK'
    try: c.at[i, 'v1_lin_semcor'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v1_lin_semcor'] = 'UNK'
    try: c.at[i, 'v2_lin_semcor'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v2_lin_semcor'] = 'UNK'
    try: c.at[i, 'puntuacion_lw_lin_semcor'] = similitud_frase_lw(c.at[i, 'v1_lin_semcor'], c.at[i, 'v2_lin_semcor'] )
    except:  c.at[i, 'puntuacion_lw_lin_semcor'] = 'UNK'
    try: c.at[i, 'v1_res_genesis'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v1_res_genesis'] = 'UNK'
    try: c.at[i, 'v2_res_genesis'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v2_res_genesis'] = 'UNK'
    try: c.at[i, 'puntuacion_lw_res_genesis'] = similitud_frase_lw(c.at[i, 'v1_res_genesis'], c.at[i, 'v2_res_genesis'] )
    except:  c.at[i, 'puntuacion_lw_res_genesis'] = 'UNK'
    try: c.at[i, 'v1_jcn_genesis'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v1_jcn_genesis'] = 'UNK'
    try: c.at[i, 'v2_jcn_genesis'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v2_jcn_genesis'] = 'UNK'
    try: c.at[i, 'puntuacion_lw_jcn_genesis'] = similitud_frase_lw(c.at[i, 'v1_jcn_genesis'], c.at[i, 'v2_jcn_genesis'] )
    except:  c.at[i, 'puntuacion_lw_jcn_genesis'] = 'UNK'
    try: c.at[i, 'v1_lin_genesis'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v1_lin_genesis'] = 'UNK'
    try: c.at[i, 'v2_lin_genesis'] = v_semantico(c.at[i, 'syns1'], c.at[i, 'bow'], medida='lw', ic='')
    except:  c.at[i, 'v2_lin_genesis'] = 'UNK'
    try: c.at[i, 'puntuacion_lw_lin_genesis'] = similitud_frase_lw(c.at[i, 'v1_lin_genesis'], c.at[i, 'v2_lin_genesis'] )
    except:  c.at[i, 'puntuacion_lw_lin_genesis'] = 'UNK'
    if print_info == 'y':
        print('v1_lw', i, ': ', c.at[i, 'v1_lw'])
        print('v2_lw', i, ': ', c.at[i, 'v2_lw'])
        print('puntuacion_lw_lw', i, ': ', c.at[i, 'puntuacion_lw_lw'])
        print()    
'''

In [ ]:
# Algunos análisis previos a la creación de las agrupaciones syns1 y syns2 extendidas

syns = c['syns1'][0]
print()
syn = c['syns1'][0][0]
print()
print('syns:', syns)
print()
print('syn:', syn)
print()
print(syn.definition())
print('Hyponyms:', syn.hyponyms())
print()
print('hypernyms:', syn.hypernyms())
print()

'''
Otras relaciones que no tenemos en cuenta para establecer relevancia

print('instance_hyponyms:', syn.instance_hyponyms())
print()
print('instance_hyponyms:', syn.instance_hyponyms())
print()
print('troponyms:', syn.troponyms())
print()
print('member_holonyms:', syn.member_holonyms())
print()
print('substance_holonyms:', syn.substance_holonyms())
print()
print('part_holonyms:', syn.part_holonyms())
print()
print('member_meronyms:', syn.member_meronyms())
print()
print('substance_meronyms:', syn.substance_meronyms())
print()
print('part_meronyms:', syn.part_meronyms())
print()
'''


In [ ]:
# Expansión de los conjuntos syns1 y syns2 para incluir los nodos de relevancia
    # Para cada synset-nombre añadimos todos los hiperónimos y homónimos
    # Para cada synset-verbo añadimos los hiperónimos y tropónimmos

syns_extendido = syns.copy()
print('syns_extendido:', syns_extendido)
print()

for synset in syns:
    print('SYNSET:', synset)
    print('HYPERÓNIMOS:', synset.hypernyms())
    print('HYPÓNIMOS:', synset.hyponyms())
    syns_extendido.extend(synset.hypernyms())
    syns_extendido.extend(synset.hyponyms())
    print('SYNS_EXTENDIDO:', syns_extendido)
    print()

In [ ]:
# Expansión de los conjuntos syns1 y syns2 para incluir los nodos de relevancia (hiperónimos y homónimos)

def syns_extendido(syns):
    syns_extendido = syns.copy()
    print('syns_extendido:', syns_extendido)
    print()

    for synset in syns:
        print('SYNSET:', synset)
        print('HYPERÓNIMOS:', synset.hypernyms())
        print('HYPÓNIMOS:', synset.hyponyms())
        syns_extendido.extend(synset.hypernyms())
        syns_extendido.extend(synset.hyponyms())
        print('SYNS_EXTENDIDO:', syns_extendido)
        print()
    
    return(syns_extendido)
        
# Creación y almacenamiento de los synsets extendidos

c['syns1_ext'] = ''
c['syns2_ext'] = ''

for i in c.index:
    c.at[i, 'syns1_ext'] = syns_extendido(c.at[i,'syns1'])
    c.at[i, 'syns2_ext'] = syns_extendido(c.at[i,'syns2'])
    
    
        

In [ ]:
c.head()

# Análisis 3
## Errores según:
* Desambiguador de Lesk
* Función de similitud aplicada
    * Todas las funciones de NLTK: solo para nombres y verbos.
    * Similitud de Resnik: 
        * Parece que cuando un synset c1 no está en el corpus (ej. c1=Synset('leek.n.01')
            * sim_res (c1,c1) = infinito (10 elevado a 300)
            * sim_res (c1, Synset('woman.n.01')) = 2,22.
        * Cuando los synsets sí están en el corpus, da un valor finito aunque sean iguales.
    * Similitud de Jiang-Ch: 
        * Cuando las palabras son iguales, la similitud es infinito (10 elevado a 300) --> PONER UN LÍMITE.
        

## Éxito según:
* Tipo de texto
* Número de palabras por frase (¿más éxito en frases largas?)


## BoW amb n+v / n+v+a+r

In [ ]:
# ==================================
# 10. ANÁLISIS DE RESULTADOS
# ==================================

# 10.1 ERRORES

from nltk.corpus import wordnet_ic
from nltk.corpus import brown

brown_ic = wordnet_ic.ic('ic-brown.dat')
brown_vocab = set(sorted([word for word in brown.words()]))

woman = ('woman' in brown_vocab)
leek = ('leek' in brown_vocab)
print("Está 'woman' en el corpus de Brown?", woman)
print("Está 'leek' en el corpus de Brown?", leek)


In [ ]:
# Similitud de Resnik

print("wn.res_similarity(wn.synset('leek.n.01'), wn.synset('leek.n.01'), brown_ic):", wn.res_similarity(wn.synset('leek.n.01'), wn.synset('leek.n.01'), brown_ic))
print("wn.res_similarity(wn.synset('leek.n.01'), wn.synset('woman.n.01'), brown_ic):", wn.res_similarity(wn.synset('leek.n.01'), wn.synset('woman.n.01'), brown_ic))


In [ ]:
# Similitud Jiang-Ch

print(wn.jcn_similarity(wn.synset('leek.n.01'), wn.synset('woman.n.01'), brown_ic))
print(wn.jcn_similarity(wn.synset('woman.n.01'), wn.synset('woman.n.01'), brown_ic))
print(wn.jcn_similarity(wn.synset('leek.n.01'), wn.synset('leek.n.01'), brown_ic))
print(wn.jcn_similarity(wn.synset('woman.n.01'), wn.synset('girl.n.01'), brown_ic))

In [ ]:
# función de similitud según [Liu 2013]
    # Nuestro BoW és un "Bag of Synsets": se han excluído las palabras que no está en Wordnet        
'''
    PENDIENTE: Reproducir exactamente [Liu 2013] incluyendo en el espacio vectorial las palabras que no están en Wordnet, 
    y asignarles similitud 1 si coinciden y similitud cero si no coinciden.
'''

syn = c['syns1'][0][0]
print('syn:', syn)
bow = c['bow'][0]
print('bow:', bow)

v = np.zeros(len(bow))
print('v:', v)
print()

for i in enumerate(bow):
    print('enumerate(bow):', i)
    print('v[i[0]]:', v[i[0]])
    print('syn:', syn)
    print('i[1]:', i[1])
    if syn == i[1]:    # "If the two target words are identical or in the same synset, then the similarity is 1"
        v[i[0]] = 1
    print('v[i[0]]:', v[i[0]])        
    print()

In [ ]:
# Concept vectors of Wordnet nodes following [Liu 2013] aproximation

syn = c['syns1'][0][0]
print('syn:', syn)

In [ ]:
type(wn.all_synsets())

In [ ]:
# Iteración lenta si solo queremos ver unos pocos ejemplos

for synset in list(wn.all_synsets())[:10]:
    print(synset)
    
print()



In [ ]:
# Iteración rápida, pero requiere la herramienta itertools

from itertools import islice

for synset in islice(wn.all_synsets(), 10):
    print(synset)

In [ ]:
# Otra manera de iterar rápido los primeros ejemplos de un conjunto grande:

for index, synset in zip(range(10), wn.all_synsets()):
    print(synset)


In [ ]:
for synset in islice(wn.all_synsets('n'), 10):
    print(synset, synset.min_depth())

In [ ]:
wn_nombres = [synset for synset in wn.all_synsets('n')]

In [ ]:
wn_nombres = sorted(wn_nombres, key=lambda wn_nombre: wn_nombre.max_depth())

In [ ]:
for wn_nombre in wn_nombres:
    print(wn_nombre, wn_nombre.max_depth())

In [ ]:
ordenados = sorted(wn_nombres, key=min_depth())

In [ ]:
wn_nombres[1].min_depth()

In [ ]:
wn_nombres2 = sorted([synset for synset in wn.all_synsets('n')], key=synset.min_depth())